# Phase 3A: Teacher Pool Entry Gate

Notebook này là cổng chuyển chính thức từ Phase 2 sang Phase 3.

Notebook không huấn luyện Teacher. Nó thực hiện:

- xác minh Phase 1 Formal Output;
- xác minh Phase 2A Shared Training Bundle;
- xác minh đủ 9 baseline run: S1/S2/S3 × seed 42/123/2026;
- kiểm tra checkpoint, manifest, log và Validation metrics;
- ghi nhận protocol deviation của S3 seed 123;
- tổng hợp mean ± sample standard deviation;
- khóa matched baseline cho ba Teacher family;
- tạo `PHASE2_STUDENT_BASELINES_COMPLETE.json`;
- tạo `PHASE3_TEACHER_POOL_INPUT_READY.json`;
- tiếp tục giữ Held-out Test Set ở trạng thái `SEALED`.

## Kaggle inputs bắt buộc

Gắn đúng một version output cho mỗi nguồn sau:

1. Phase 1 — MS COCO Data Contract.
2. Phase 2A — Shared Student Pipeline.
3. S1 seed 42.
4. S1 seed 123.
5. S1 seed 2026.
6. S2 seed 42.
7. S2 seed 123.
8. S2 seed 2026.
9. S3 seed 42.
10. S3 seed 123.
11. S3 seed 2026.

Không gắn nhiều version của cùng một run.


## Cell 1 — Môi trường, thư mục output và khám phá input

### Công việc

- khóa seed 42 cho thao tác tổng hợp;
- tạo output root của Phase 3A;
- tìm READY marker Phase 1 và Phase 2A;
- tìm chính xác chín COMPLETE marker baseline;
- kiểm tra đủ ma trận Student × seed;
- chưa mở Parquet, checkpoint hoặc Test Set.

### Thời gian dự kiến

5–20 giây.

### Đầu ra dự kiến

- `PHASE 3A CELL 1 STATUS: PASSED`;
- Phase 1 marker: 1;
- Phase 2A marker: 1;
- baseline COMPLETE marker: 9;
- ma trận S1/S2/S3 × 42/123/2026 đầy đủ.


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import random
import statistics
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
BASE_SEED = 42
EXPECTED_STUDENTS = {
    "S1": "MobileNetV3-Large",
    "S2": "MobileViT-S",
    "S3": "TinyViT-5M",
}
EXPECTED_SEEDS = {42, 123, 2026}

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = (
    KAGGLE_WORKING_ROOT
    / ""
    / "phase3_teacher_pool_entry"
)
CONFIG_DIR = OUTPUT_ROOT / "configs"
AUDIT_DIR = OUTPUT_ROOT / "audits"
RESULT_DIR = OUTPUT_ROOT / "results"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"

for directory in [
    OUTPUT_ROOT,
    CONFIG_DIR,
    AUDIT_DIR,
    RESULT_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def atomic_write_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def write_json(path, payload):
    atomic_write_text(
        path,
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
    )


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        while True:
            chunk = source.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root), (
        f"Path escapes root: {relative_path}"
    )
    return candidate


random.seed(BASE_SEED)
assert KAGGLE_INPUT_ROOT.is_dir(), (
    "Notebook này phải chạy trên Kaggle."
)

phase1_markers = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE1_FORMAL_DELIVERABLES_READY.json"
    )
)
phase2a_markers = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
    )
)
baseline_markers = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "*_FULL_RUN_COMPLETE.json"
    )
)

assert len(phase1_markers) == 1, (
    "Cần gắn đúng một Phase 1 Formal Output; "
    f"tìm thấy {len(phase1_markers)}."
)
assert len(phase2a_markers) == 1, (
    "Cần gắn đúng một Phase 2A Shared Bundle; "
    f"tìm thấy {len(phase2a_markers)}."
)
assert len(baseline_markers) == 9, (
    "Cần gắn đúng chín baseline outputs; "
    f"tìm thấy {len(baseline_markers)}."
)

discovered_matrix = set()
discovered_marker_records = []
for marker_path in baseline_markers:
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    student_id = marker["student_id"]
    seed = int(marker["seed"])
    assert student_id in EXPECTED_STUDENTS
    assert marker["student_name"] == EXPECTED_STUDENTS[student_id]
    assert seed in EXPECTED_SEEDS
    key = (student_id, seed)
    assert key not in discovered_matrix, (
        f"Duplicate baseline marker: {key}"
    )
    discovered_matrix.add(key)
    discovered_marker_records.append(
        {
            "student_id": student_id,
            "student_name": marker["student_name"],
            "seed": seed,
            "marker_path": str(marker_path),
        }
    )

expected_matrix = {
    (student_id, seed)
    for student_id in EXPECTED_STUDENTS
    for seed in EXPECTED_SEEDS
}
assert discovered_matrix == expected_matrix

cell_audit = {
    "schema_version": "phase3a_cell01_discovery_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "seed": BASE_SEED,
    },
    "discovery": {
        "phase1_markers": len(phase1_markers),
        "phase2a_markers": len(phase2a_markers),
        "baseline_complete_markers": len(baseline_markers),
        "baseline_matrix": discovered_marker_records,
    },
    "data_policy": {
        "parquet_opened": False,
        "checkpoint_opened": False,
        "test_set_opened": False,
        "test_set_used": False,
    },
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
CELL01_AUDIT_PATH = AUDIT_DIR / "phase3a_cell01_input_discovery_audit.json"
write_json(CELL01_AUDIT_PATH, cell_audit)

print("=" * 72)
print("PHASE 3A INPUT DISCOVERY")
print("=" * 72)
print("Python                    :", platform.python_version())
print("Phase 1 READY markers     :", len(phase1_markers))
print("Phase 2A READY markers    :", len(phase2a_markers))
print("Baseline COMPLETE markers :", len(baseline_markers))
for student_id in sorted(EXPECTED_STUDENTS):
    seeds = sorted(
        seed
        for sid, seed in discovered_matrix
        if sid == student_id
    )
    print(
        f"{student_id} seeds                 :",
        seeds,
    )

print()
print("=" * 72)
print("PHASE 3A CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Baseline matrix           : 3 students x 3 seeds")
print("Test Set opened           : False")
print("Test Set used             : False")
print("Audit                     :", CELL01_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_audit["elapsed_seconds"], 2),
    "giây",
)
print(
    "Next                      : verify Phase 1 and Phase 2A contracts"
)


PHASE 3A INPUT DISCOVERY
Python                    : 3.12.13
Phase 1 READY markers     : 1
Phase 2A READY markers    : 1
Baseline COMPLETE markers : 9
S1 seeds                 : [42, 123, 2026]
S2 seeds                 : [42, 123, 2026]
S3 seeds                 : [42, 123, 2026]

PHASE 3A CELL 1 STATUS
Status                    : PASSED
Baseline matrix           : 3 students x 3 seeds
Test Set opened           : False
Test Set used             : False
Audit                     : /kaggle/working//phase3_teacher_pool_entry/audits/phase3a_cell01_input_discovery_audit.json
Elapsed                   : 0.89 giây
Next                      : verify Phase 1 and Phase 2A contracts


## Cell 2 — Xác minh Phase 1 và Phase 2A

### Công việc

- xác minh hash của Phase 1 `SHA256SUMS`;
- xác minh các development artifact bắt buộc mà không mở Test Parquet;
- xác minh 38 artefact trong Phase 2A Shared Bundle;
- tải shared training contract;
- khóa dataset, split, vocabulary, decoder và overflow policy gốc;
- không đọc caption hoặc ảnh Test.

### Thời gian dự kiến

20–90 giây, tùy tốc độ đọc Kaggle Input.

### Đầu ra dự kiến

- Phase 1 Formal READY: VERIFIED;
- Phase 2A artefacts: 38/38;
- dataset: MS COCO 2014 only;
- split: Karpathy;
- shared overflow limit: 10;
- Test Set: SEALED.


In [2]:
CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "phase1_markers",
    "phase2a_markers",
    "sha256_file",
    "read_json",
    "write_json",
    "resolve_inside",
    "AUDIT_DIR",
]
assert not [
    name for name in required_runtime_names
    if name not in globals()
], "Hãy chạy Cell 1 trước."

PHASE1_READY_PATH = phase1_markers[0].resolve()
PHASE1_ROOT = PHASE1_READY_PATH.parent.resolve()
phase1_ready = read_json(PHASE1_READY_PATH)
assert phase1_ready["status"] == "ready"
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False
assert phase1_ready["flickr_used"] is False

PHASE1_SHA256SUMS_PATH = (
    PHASE1_ROOT / "SHA256SUMS.txt"
)
assert PHASE1_SHA256SUMS_PATH.is_file()
assert sha256_file(PHASE1_SHA256SUMS_PATH) == (
    phase1_ready["sha256sums_sha256"]
)

phase1_checksums = {}
for raw_line in PHASE1_SHA256SUMS_PATH.read_text(
    encoding="utf-8"
).splitlines():
    if not raw_line.strip():
        continue
    checksum, relative_path = raw_line.split("  ", 1)
    assert len(checksum) == 64
    assert relative_path not in phase1_checksums
    phase1_checksums[relative_path] = checksum.lower()

required_phase1_development_artifacts = [
    "configs/experiment_contract_v2.json",
    "parquet/mscoco_image_manifest.parquet",
    "parquet/mscoco_train.parquet",
    "parquet/mscoco_validation.parquet",
    "vocabulary/word_vocabulary_minfreq5.json",
    "vocabulary/tokenizer_contract.json",
    "vocabulary/sequence_encoding_contract.json",
]
for relative_path in required_phase1_development_artifacts:
    artifact_path = resolve_inside(
        PHASE1_ROOT,
        relative_path,
    )
    assert artifact_path.is_file()
    assert artifact_path.stat().st_size > 0
    assert relative_path in phase1_checksums
    assert sha256_file(artifact_path) == (
        phase1_checksums[relative_path]
    )

experiment_contract = read_json(
    PHASE1_ROOT
    / "configs"
    / "experiment_contract_v2.json"
)
assert experiment_contract["schema_version"] == "2.0.0"
assert experiment_contract["project"][
    "primary_dataset_only"
] is True
assert experiment_contract["dataset"]["name"] == "MS COCO"
assert experiment_contract["dataset"]["release"] == "2014"
assert experiment_contract["dataset"]["split_protocol"] == "Karpathy"

PHASE2A_READY_PATH = phase2a_markers[0].resolve()
PHASE2A_ROOT = PHASE2A_READY_PATH.parent.resolve()
phase2a_ready = read_json(PHASE2A_READY_PATH)
assert phase2a_ready["status"] == "ready"
assert phase2a_ready["relocatable"] is True
assert phase2a_ready["artifact_count"] == 38
assert phase2a_ready["data_policy"]["test_set_status"] == "sealed"
assert phase2a_ready["data_policy"]["test_set_used"] is False

phase2a_manifest_paths = {}
for name, record in phase2a_ready["manifests"].items():
    path = resolve_inside(
        PHASE2A_ROOT,
        record["relative_path"],
    )
    assert path.is_file()
    assert sha256_file(path) == record["sha256"]
    phase2a_manifest_paths[name] = path

with open(
    phase2a_manifest_paths["csv"],
    "r",
    encoding="utf-8",
    newline="",
) as source:
    phase2a_manifest_rows = list(csv.DictReader(source))

assert len(phase2a_manifest_rows) == 38
for row in phase2a_manifest_rows:
    path = resolve_inside(
        PHASE2A_ROOT,
        row["relative_path"],
    )
    assert path.is_file()
    assert int(row["size_bytes"]) == path.stat().st_size
    assert sha256_file(path) == row["sha256"]

SHARED_TRAINING_CONTRACT_PATH = (
    PHASE2A_ROOT
    / "configs"
    / "phase2_shared_baseline_training_contract.json"
)
shared_training_contract = read_json(
    SHARED_TRAINING_CONTRACT_PATH
)
assert shared_training_contract["status"] == "locked"
assert shared_training_contract["scope"][
    "primary_dataset"
] == "MS COCO 2014 only"
assert shared_training_contract["scope"][
    "split_protocol"
] == "Karpathy"
assert shared_training_contract["scope"][
    "test_set_status"
] == "sealed"

SHARED_OVERFLOW_LIMIT = int(
    shared_training_contract["optimization"][
        "mixed_precision"
    ]["maximum_overflow_retries_per_run"]
)
assert SHARED_OVERFLOW_LIMIT == 10

cell_audit = {
    "schema_version": "phase3a_cell02_upstream_contract_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "phase1": {
        "ready_marker": str(PHASE1_READY_PATH),
        "sha256sums_rows": len(phase1_checksums),
        "development_artifacts_verified": len(
            required_phase1_development_artifacts
        ),
        "test_parquet_opened": False,
    },
    "phase2a": {
        "ready_marker": str(PHASE2A_READY_PATH),
        "manifest_rows_verified": len(
            phase2a_manifest_rows
        ),
        "shared_training_contract_sha256": sha256_file(
            SHARED_TRAINING_CONTRACT_PATH
        ),
        "shared_overflow_limit": SHARED_OVERFLOW_LIMIT,
    },
    "data_policy": {
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
    },
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
CELL02_AUDIT_PATH = AUDIT_DIR / (
    "phase3a_cell02_upstream_contract_audit.json"
)
write_json(CELL02_AUDIT_PATH, cell_audit)

print("=" * 72)
print("UPSTREAM CONTRACT VERIFICATION")
print("=" * 72)
print("Phase 1 Formal READY      : VERIFIED")
print(
    "Phase 1 development files:",
    len(required_phase1_development_artifacts),
    "/ 7 VERIFIED",
)
print("Phase 2A bundle           : VERIFIED")
print(
    "Phase 2A manifest rows   :",
    len(phase2a_manifest_rows),
    "/ 38 VERIFIED",
)
print("Dataset                   : MS COCO 2014 only")
print("Split                     : Karpathy")
print("Shared overflow limit     :", SHARED_OVERFLOW_LIMIT)
print("Test Set status           : SEALED")

print()
print("=" * 72)
print("PHASE 3A CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Audit                     :", CELL02_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_audit["elapsed_seconds"], 2),
    "giây",
)
print(
    "Next                      : verify all nine baseline run bundles"
)


UPSTREAM CONTRACT VERIFICATION
Phase 1 Formal READY      : VERIFIED
Phase 1 development files: 7 / 7 VERIFIED
Phase 2A bundle           : VERIFIED
Phase 2A manifest rows   : 38 / 38 VERIFIED
Dataset                   : MS COCO 2014 only
Split                     : Karpathy
Shared overflow limit     : 10
Test Set status           : SEALED

PHASE 3A CELL 2 STATUS
Status                    : PASSED
Test Parquet opened       : False
Test caption read         : False
Test images read          : False
Audit                     : /kaggle/working//phase3_teacher_pool_entry/audits/phase3a_cell02_upstream_contract_audit.json
Elapsed                   : 2.11 giây
Next                      : verify all nine baseline run bundles


## Cell 3 — Xác minh đầy đủ chín baseline run

### Công việc

- kiểm tra COMPLETE marker của từng Student–seed;
- kiểm tra best/last checkpoint;
- kiểm tra manifest CSV, manifest JSON và SHA256SUMS;
- hash lại mọi artefact được liệt kê;
- đọc Cell 11 audit và loaded-model contract;
- thu thập params, thời gian, overflow và Validation metrics;
- không load trọng số checkpoint vào PyTorch;
- không mở Test Parquet.

### Thời gian dự kiến

2–8 phút vì phải đọc và hash khoảng 2,7 GiB.

### Đầu ra dự kiến

- baseline bundles: 9/9 VERIFIED;
- manifest artefacts: khoảng 431 dòng;
- core hashes: 45/45;
- Test Set: SEALED.


In [3]:
CELL_STARTED_AT = time.perf_counter()

assert "baseline_markers" in globals()
assert "SHARED_OVERFLOW_LIMIT" in globals()


def find_unique_by_name(root, name):
    matches = list(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Expected one {name}, found {len(matches)} in {root}"
    )
    return matches[0]


def find_unique_by_pattern(root, pattern):
    matches = list(Path(root).rglob(pattern))
    assert len(matches) == 1, (
        f"Expected one {pattern}, found {len(matches)} in {root}"
    )
    return matches[0]


run_records = []
total_manifest_rows = 0
core_hashes_verified = 0

for marker_path in baseline_markers:
    marker_path = marker_path.resolve()
    run_root = marker_path.parent.resolve()
    marker = read_json(marker_path)
    student_id = marker["student_id"]
    seed = int(marker["seed"])

    assert marker["status"] == "complete"
    assert marker["student_name"] == EXPECTED_STUDENTS[student_id]
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    core_specs = [
        (
            "best_checkpoint",
            marker["best_checkpoint"]["path"],
            marker["best_checkpoint"]["sha256"],
        ),
        (
            "last_checkpoint",
            marker["last_checkpoint"]["path"],
            marker["last_checkpoint"]["sha256"],
        ),
        (
            "manifest_csv",
            marker["manifest"]["csv"]["path"],
            marker["manifest"]["csv"]["sha256"],
        ),
        (
            "manifest_json",
            marker["manifest"]["json"]["path"],
            marker["manifest"]["json"]["sha256"],
        ),
        (
            "sha256sums",
            marker["manifest"]["sha256sums"]["path"],
            marker["manifest"]["sha256sums"]["sha256"],
        ),
    ]
    resolved_core = {}
    for kind, original_path, expected_hash in core_specs:
        path = find_unique_by_name(
            run_root,
            Path(original_path).name,
        )
        assert sha256_file(path) == expected_hash
        resolved_core[kind] = path
        core_hashes_verified += 1

    with open(
        resolved_core["manifest_csv"],
        "r",
        encoding="utf-8",
        newline="",
    ) as source:
        manifest_rows = list(csv.DictReader(source))
    assert len(manifest_rows) == int(
        marker["manifest_artifact_count"]
    )

    manifest_json = read_json(
        resolved_core["manifest_json"]
    )
    assert manifest_json["artifact_count"] == len(
        manifest_rows
    )
    assert manifest_json["test_set_status"] == "sealed"
    assert manifest_json["test_set_used"] is False

    for row in manifest_rows:
        path = resolve_inside(
            run_root,
            row["relative_path"],
        )
        assert path.is_file()
        assert path.stat().st_size == int(row["size_bytes"])
        assert sha256_file(path) == row["sha256"]
    total_manifest_rows += len(manifest_rows)

    audit_path = find_unique_by_pattern(
        run_root,
        f"{student_id.lower()}_cell11_full_seed{seed}_training_run_audit.json",
    )
    audit = read_json(audit_path)
    assert audit["status"] == "passed"
    assert int(audit["seed"]) == seed
    assert audit["student_id"] == student_id
    assert audit["test_parquet_opened"] is False
    assert audit["test_caption_read"] is False
    assert audit["test_images_read"] is False
    assert audit["test_predictions_written"] is False
    assert audit["test_metrics_computed"] is False
    assert audit["test_set_status"] == "sealed"
    assert audit["test_set_used"] is False
    assert audit["flickr_used"] is False

    for metric_name, value in marker[
        "best_validation_metrics"
    ].items():
        assert math.isfinite(float(value))
        assert math.isclose(
            float(value),
            float(audit["best_metrics"][metric_name]),
            rel_tol=0.0,
            abs_tol=0.0,
        )

    model_contract_path = find_unique_by_pattern(
        run_root,
        f"{student_id.lower()}_loaded_model_contract.json",
    )
    model_contract = read_json(model_contract_path)
    if "architecture" in model_contract:
        parameters = int(
            model_contract["architecture"][
                "end_to_end_parameters"
            ]
        )
    else:
        parameters = int(
            model_contract["end_to_end_parameters"]
        )

    trainer_contract_path = find_unique_by_pattern(
        run_root,
        f"{student_id.lower()}_production_trainer_checkpoint_early_stop_contract.json",
    )
    trainer_contract = read_json(trainer_contract_path)
    run_overflow_limit = int(
        trainer_contract["epoch_trainer"][
            "maximum_overflow_retries"
        ]
    )

    optimizer_log_path = find_unique_by_pattern(
        run_root,
        f"{student_id.lower()}_seed{seed}_optimizer_steps.jsonl",
    )
    epoch_history_path = find_unique_by_pattern(
        run_root,
        f"{student_id.lower()}_seed{seed}_epoch_history.csv",
    )

    run_records.append(
        {
            "student_id": student_id,
            "student_name": marker["student_name"],
            "seed": seed,
            "completed_epochs": int(
                marker["completed_epochs"]
            ),
            "global_optimizer_steps": int(
                marker["global_optimizer_steps"]
            ),
            "best_epoch": int(marker["best_epoch"]),
            "validation_ce": float(
                marker["best_validation_cross_entropy"]
            ),
            "bleu_1": float(
                marker["best_validation_metrics"]["BLEU-1"]
            ),
            "bleu_4": float(
                marker["best_validation_metrics"]["BLEU-4"]
            ),
            "meteor": float(
                marker["best_validation_metrics"]["METEOR"]
            ),
            "rouge_l": float(
                marker["best_validation_metrics"]["ROUGE-L"]
            ),
            "cider_d": float(
                marker["best_validation_metrics"]["CIDEr-D"]
            ),
            "parameters": parameters,
            "training_seconds": float(
                audit["training_elapsed_seconds"]
            ),
            "overflow_retries": int(
                audit["overflow_retry_count"]
            ),
            "run_overflow_limit": run_overflow_limit,
            "manifest_rows": len(manifest_rows),
            "marker_path": str(marker_path),
            "run_root": str(run_root),
            "audit_path": str(audit_path),
            "trainer_contract_path": str(
                trainer_contract_path
            ),
            "optimizer_log_path": str(
                optimizer_log_path
            ),
            "epoch_history_path": str(
                epoch_history_path
            ),
            "best_checkpoint_path": str(
                resolved_core["best_checkpoint"]
            ),
            "best_checkpoint_sha256": marker[
                "best_checkpoint"
            ]["sha256"],
            "last_checkpoint_path": str(
                resolved_core["last_checkpoint"]
            ),
            "last_checkpoint_sha256": marker[
                "last_checkpoint"
            ]["sha256"],
            "test_set_status": "sealed",
            "test_set_used": False,
        }
    )

run_records.sort(
    key=lambda row: (row["student_id"], row["seed"])
)
assert len(run_records) == 9
assert core_hashes_verified == 45

RUN_REGISTRY_PATH = CONFIG_DIR / (
    "phase2_nine_run_verified_registry.json"
)
write_json(
    RUN_REGISTRY_PATH,
    {
        "schema_version": "phase2_nine_run_registry_v1",
        "created_at_utc": now_utc(),
        "status": "verified",
        "runs": run_records,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit = {
    "schema_version": "phase3a_cell03_baseline_bundle_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "run_count": len(run_records),
    "core_hashes_verified": core_hashes_verified,
    "manifest_rows_verified": total_manifest_rows,
    "run_registry": str(RUN_REGISTRY_PATH),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
CELL03_AUDIT_PATH = AUDIT_DIR / (
    "phase3a_cell03_nine_baseline_bundle_audit.json"
)
write_json(CELL03_AUDIT_PATH, cell_audit)

print("=" * 72)
print("NINE BASELINE RUN BUNDLE VERIFICATION")
print("=" * 72)
for row in run_records:
    print(
        f"{row['student_id']} seed {row['seed']:4d} : "
        f"VERIFIED | artifacts={row['manifest_rows']:2d} | "
        f"CIDEr-D={row['cider_d']:.8f}"
    )
print()
print("Run bundles              :", len(run_records), "/ 9")
print("Core hashes              :", core_hashes_verified, "/ 45")
print("Manifest artefacts       :", total_manifest_rows)

print()
print("=" * 72)
print("PHASE 3A CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Baseline runs             : 9 / 9 VERIFIED")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Audit                     :", CELL03_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_audit["elapsed_seconds"], 2),
    "giây",
)
print(
    "Next                      : audit optimizer logs and protocol deviations"
)


NINE BASELINE RUN BUNDLE VERIFICATION
S1 seed   42 : VERIFIED | artifacts=48 | CIDEr-D=0.99179116
S1 seed  123 : VERIFIED | artifacts=48 | CIDEr-D=0.98725076
S1 seed 2026 : VERIFIED | artifacts=47 | CIDEr-D=0.97845069
S2 seed   42 : VERIFIED | artifacts=48 | CIDEr-D=0.99735544
S2 seed  123 : VERIFIED | artifacts=44 | CIDEr-D=0.98238114
S2 seed 2026 : VERIFIED | artifacts=48 | CIDEr-D=0.99993979
S3 seed   42 : VERIFIED | artifacts=48 | CIDEr-D=1.07679862
S3 seed  123 : VERIFIED | artifacts=48 | CIDEr-D=1.07671028
S3 seed 2026 : VERIFIED | artifacts=47 | CIDEr-D=1.07090187

Run bundles              : 9 / 9
Core hashes              : 45 / 45
Manifest artefacts       : 426

PHASE 3A CELL 3 STATUS
Status                    : PASSED
Baseline runs             : 9 / 9 VERIFIED
Test Parquet opened       : False
Test images read          : False
Test metrics computed     : False
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_pool_entry/audits/phase

## Cell 4 — Audit optimizer log và protocol deviation

### Công việc

- đọc toàn bộ optimizer-step JSONL của chín run;
- kiểm tra JSON hợp lệ, loss/gradient hữu hạn;
- đếm successful step và overflow retry;
- xác minh Test Set không được dùng;
- kiểm tra epoch-history và validation artefact;
- phát hiện sai lệch giữa shared contract và run-local contract;
- chấp nhận có khai báo đúng một deviation đã biết: S3 seed 123, limit 32, actual 11.

### Thời gian dự kiến

15–60 giây.

### Đầu ra dự kiến

- khoảng 308.154 log rows;
- 308.080 successful steps;
- 74 overflow rows;
- NaN/Inf: 0;
- protocol deviations: 1;
- Test-used rows: 0.


In [4]:
CELL_STARTED_AT = time.perf_counter()

assert "run_records" in globals()
assert len(run_records) == 9

total_log_rows = 0
total_successful = 0
total_overflows = 0
total_nonfinite = 0
total_test_used_rows = 0
protocol_deviations = []

for row in run_records:
    log_path = Path(row["optimizer_log_path"])
    log_rows = 0
    successful = 0
    overflows = 0
    nonfinite = 0
    test_used_rows = 0

    with open(log_path, "r", encoding="utf-8") as source:
        for line_number, line in enumerate(source, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            log_rows += 1
            if bool(record["successful_step"]):
                successful += 1
            else:
                overflows += 1
            if bool(record["test_set_used"]):
                test_used_rows += 1

            numeric_values = [
                record["mean_token_cross_entropy"],
                record["gradient_norm_before_clip"],
                record["grad_scaler_scale_before"],
                record["grad_scaler_scale_after"],
            ]
            numeric_values.extend(
                record["learning_rates"].values()
            )
            if not all(
                math.isfinite(float(value))
                for value in numeric_values
            ):
                nonfinite += 1

    assert successful == row["global_optimizer_steps"]
    assert overflows == row["overflow_retries"]
    assert nonfinite == 0
    assert test_used_rows == 0

    with open(
        row["epoch_history_path"],
        "r",
        encoding="utf-8",
        newline="",
    ) as source:
        history_rows = list(csv.DictReader(source))
    assert len(history_rows) == row["completed_epochs"]

    validation_directory = (
        Path(row["run_root"])
        / "validation"
        / "epochs"
    )
    validation_files = sorted(
        validation_directory.glob(
            "epoch_*_validation.json"
        )
    )
    assert len(validation_files) == row["completed_epochs"]

    total_log_rows += log_rows
    total_successful += successful
    total_overflows += overflows
    total_nonfinite += nonfinite
    total_test_used_rows += test_used_rows

    if (
        row["run_overflow_limit"]
        != SHARED_OVERFLOW_LIMIT
    ):
        protocol_deviations.append(
            {
                "deviation_id": "P2-DEV-001",
                "student_id": row["student_id"],
                "student_name": row["student_name"],
                "seed": row["seed"],
                "classification": (
                    "operational safety-gate deviation"
                ),
                "status": "accepted_with_disclosure",
                "shared_overflow_limit": (
                    SHARED_OVERFLOW_LIMIT
                ),
                "run_local_overflow_limit": (
                    row["run_overflow_limit"]
                ),
                "observed_overflow_retries": (
                    row["overflow_retries"]
                ),
                "successful_optimizer_steps": successful,
                "nonfinite_successful_rows": nonfinite,
                "test_set_status": "sealed",
                "test_set_used": False,
                "impact": (
                    "The changed value is an abort threshold. "
                    "Overflow rows skipped optimizer and scheduler "
                    "advancement. The run remains usable but is not "
                    "strictly identical to shared contract v2."
                ),
                "resolution": (
                    "Preserve immutable original artifacts and hashes; "
                    "disclose the deviation in every scientific table."
                ),
            }
        )

assert total_log_rows > 0
assert total_nonfinite == 0
assert total_test_used_rows == 0
assert len(protocol_deviations) == 1
known_deviation = protocol_deviations[0]
assert known_deviation["student_id"] == "S3"
assert known_deviation["seed"] == 123
assert known_deviation["shared_overflow_limit"] == 10
assert known_deviation["run_local_overflow_limit"] == 32
assert known_deviation["observed_overflow_retries"] == 11

PROTOCOL_DEVIATION_PATH = CONFIG_DIR / (
    "phase2_protocol_deviations.json"
)
write_json(
    PROTOCOL_DEVIATION_PATH,
    {
        "schema_version": "phase2_protocol_deviations_v1",
        "created_at_utc": now_utc(),
        "status": "accepted_with_disclosure",
        "deviation_count": len(protocol_deviations),
        "deviations": protocol_deviations,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit = {
    "schema_version": "phase3a_cell04_log_deviation_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "optimizer_log_rows": total_log_rows,
    "successful_optimizer_steps": total_successful,
    "overflow_rows": total_overflows,
    "nonfinite_rows": total_nonfinite,
    "test_used_rows": total_test_used_rows,
    "protocol_deviation_count": len(
        protocol_deviations
    ),
    "protocol_deviation_path": str(
        PROTOCOL_DEVIATION_PATH
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
CELL04_AUDIT_PATH = AUDIT_DIR / (
    "phase3a_cell04_optimizer_log_and_deviation_audit.json"
)
write_json(CELL04_AUDIT_PATH, cell_audit)

print("=" * 72)
print("OPTIMIZER LOG AND PROTOCOL DEVIATION AUDIT")
print("=" * 72)
print("Optimizer log rows        :", total_log_rows)
print("Successful updates        :", total_successful)
print("Overflow rows             :", total_overflows)
print("NaN/Inf rows              :", total_nonfinite)
print("Test-used rows            :", total_test_used_rows)
print(
    "Protocol deviations       :",
    len(protocol_deviations),
)
print(
    "P2-DEV-001               : S3 seed 123 | "
    "limit 10 -> 32 | actual 11 | DISCLOSED"
)

print()
print("=" * 72)
print("PHASE 3A CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Logs finite               : True")
print("Overflow accounting       : EXACT")
print("Known deviation disclosed : True")
print("Test Set status           : SEALED")
print("Audit                     :", CELL04_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_audit["elapsed_seconds"], 2),
    "giây",
)
print(
    "Next                      : aggregate three-seed baselines and lock teacher entry targets"
)


OPTIMIZER LOG AND PROTOCOL DEVIATION AUDIT
Optimizer log rows        : 308154
Successful updates        : 308080
Overflow rows             : 74
NaN/Inf rows              : 0
Test-used rows            : 0
Protocol deviations       : 1
P2-DEV-001               : S3 seed 123 | limit 10 -> 32 | actual 11 | DISCLOSED

PHASE 3A CELL 4 STATUS
Status                    : PASSED
Logs finite               : True
Overflow accounting       : EXACT
Known deviation disclosed : True
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_pool_entry/audits/phase3a_cell04_optimizer_log_and_deviation_audit.json
Elapsed                   : 3.79 giây
Next                      : aggregate three-seed baselines and lock teacher entry targets


## Cell 5 — Tổng hợp ba seed và khóa Teacher entry contract

### Công việc

- tính mean và sample standard deviation cho sáu Validation metrics;
- xếp hạng Student theo mean CIDEr-D;
- chọn quality-first Student và deployment-efficient Student;
- tạo bảng per-run và aggregate;
- ánh xạ Teacher A/B/C với matched Student;
- khóa quy tắc logit/feature/sequence KD compatibility;
- chưa chọn Teacher checkpoint và chưa mở Test.

### Thời gian dự kiến

1–5 giây.

### Đầu ra dự kiến

- S3 đứng hạng 1 theo mean CIDEr-D;
- quality-first: S3;
- deployment-efficient: S1;
- Teacher A ↔ S1, Teacher B ↔ S2, Teacher C ↔ S3;
- Phase 3 Teacher entry contract: LOCKED.


In [5]:
CELL_STARTED_AT = time.perf_counter()

assert "run_records" in globals()
assert "protocol_deviations" in globals()

metric_names = [
    "validation_ce",
    "bleu_1",
    "bleu_4",
    "meteor",
    "rouge_l",
    "cider_d",
]
grouped = defaultdict(list)
for row in run_records:
    grouped[row["student_id"]].append(row)

aggregate_records = []
for student_id in sorted(grouped):
    rows = sorted(
        grouped[student_id],
        key=lambda row: row["seed"],
    )
    assert {row["seed"] for row in rows} == EXPECTED_SEEDS
    aggregate = {
        "student_id": student_id,
        "student_name": EXPECTED_STUDENTS[student_id],
        "seeds": [row["seed"] for row in rows],
        "n": len(rows),
        "parameters": rows[0]["parameters"],
        "training_hours_mean": statistics.mean(
            row["training_seconds"] / 3600.0
            for row in rows
        ),
        "training_hours_sample_sd": statistics.stdev(
            row["training_seconds"] / 3600.0
            for row in rows
        ),
        "overflow_total": sum(
            row["overflow_retries"]
            for row in rows
        ),
    }
    assert len({row["parameters"] for row in rows}) == 1
    for metric_name in metric_names:
        values = [
            float(row[metric_name])
            for row in rows
        ]
        aggregate[f"{metric_name}_mean"] = statistics.mean(
            values
        )
        aggregate[
            f"{metric_name}_sample_sd"
        ] = statistics.stdev(values)
        aggregate[f"{metric_name}_minimum"] = min(values)
        aggregate[f"{metric_name}_maximum"] = max(values)
    aggregate_records.append(aggregate)

ranked = sorted(
    aggregate_records,
    key=lambda row: (
        -row["cider_d_mean"],
        -row["bleu_4_mean"],
        row["validation_ce_mean"],
        row["parameters"],
    ),
)
for rank, row in enumerate(ranked, start=1):
    row["rank_by_validation_cider_d"] = rank

assert ranked[0]["student_id"] == "S3"
QUALITY_FIRST_STUDENT = ranked[0]["student_id"]
DEPLOYMENT_EFFICIENT_STUDENT = min(
    aggregate_records,
    key=lambda row: row["parameters"],
)["student_id"]
assert DEPLOYMENT_EFFICIENT_STUDENT == "S1"

PER_RUN_CSV_PATH = RESULT_DIR / (
    "phase2_three_seed_per_run.csv"
)
per_run_fields = [
    "student_id",
    "student_name",
    "seed",
    "completed_epochs",
    "best_epoch",
    "validation_ce",
    "bleu_1",
    "bleu_4",
    "meteor",
    "rouge_l",
    "cider_d",
    "parameters",
    "training_seconds",
    "overflow_retries",
    "run_overflow_limit",
    "best_checkpoint_sha256",
    "test_set_status",
    "test_set_used",
]
with open(
    PER_RUN_CSV_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=per_run_fields,
    )
    writer.writeheader()
    for row in run_records:
        writer.writerow({
            field: row[field]
            for field in per_run_fields
        })

AGGREGATE_CSV_PATH = RESULT_DIR / (
    "phase2_three_seed_aggregate.csv"
)
aggregate_fields = [
    "rank_by_validation_cider_d",
    "student_id",
    "student_name",
    "n",
    "parameters",
    "validation_ce_mean",
    "validation_ce_sample_sd",
    "bleu_1_mean",
    "bleu_1_sample_sd",
    "bleu_4_mean",
    "bleu_4_sample_sd",
    "meteor_mean",
    "meteor_sample_sd",
    "rouge_l_mean",
    "rouge_l_sample_sd",
    "cider_d_mean",
    "cider_d_sample_sd",
    "training_hours_mean",
    "training_hours_sample_sd",
    "overflow_total",
]
with open(
    AGGREGATE_CSV_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=aggregate_fields,
    )
    writer.writeheader()
    for row in ranked:
        writer.writerow({
            field: row[field]
            for field in aggregate_fields
        })

baseline_by_student_seed = {
    (row["student_id"], row["seed"]): row
    for row in run_records
}
aggregate_by_student = {
    row["student_id"]: row
    for row in aggregate_records
}

teacher_families = {
    "A": {
        "teacher": "CLIP ViT-B/32 + shared caption decoder",
        "matched_student_id": "S1",
        "matched_student": "MobileNetV3-Large",
        "default_kd_channels": [
            "logit",
            "feature",
            "sequence",
        ],
        "logit_kd_policy": (
            "allowed only with the locked shared vocabulary "
            "and identical token IDs"
        ),
    },
    "B": {
        "teacher": "BLIP caption teacher",
        "matched_student_id": "S2",
        "matched_student": "MobileViT-S",
        "default_kd_channels": [
            "feature",
            "sequence",
        ],
        "logit_kd_policy": (
            "prohibited until tokenizer/vocabulary alignment "
            "is formally demonstrated"
        ),
    },
    "C": {
        "teacher": "SigLIP ViT-B/16 + shared caption decoder",
        "matched_student_id": "S3",
        "matched_student": "TinyViT-5M",
        "default_kd_channels": [
            "logit",
            "feature",
            "sequence",
        ],
        "logit_kd_policy": (
            "allowed only after attaching the locked shared "
            "caption decoder and vocabulary"
        ),
    },
}

for family in teacher_families.values():
    student_id = family["matched_student_id"]
    seed42 = baseline_by_student_seed[
        (student_id, 42)
    ]
    aggregate = aggregate_by_student[student_id]
    family["validation_entry_targets"] = {
        "matched_seed42_cider_d": seed42["cider_d"],
        "matched_three_seed_cider_d_mean": (
            aggregate["cider_d_mean"]
        ),
        "matched_three_seed_cider_d_sample_sd": (
            aggregate["cider_d_sample_sd"]
        ),
        "teacher_pool_quality_gate": (
            "teacher Validation CIDEr-D must exceed the "
            "matched seed-42 baseline by at least 0.001, "
            "or be retained only as a documented auxiliary "
            "teacher hypothesis"
        ),
    }

TEACHER_ENTRY_CONTRACT_PATH = CONFIG_DIR / (
    "phase3_teacher_pool_entry_contract.json"
)
teacher_entry_contract = {
    "contract_version": "phase3_teacher_pool_entry_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "dataset": {
        "name": "MS COCO",
        "release": "2014",
        "split_protocol": "Karpathy",
        "training_membership": [
            "train",
            "restval",
        ],
        "validation_images": 5000,
        "held_out_test_images": 5000,
        "test_set_status": "sealed",
    },
    "selection": {
        "split": "validation",
        "primary_metric": "CIDEr-D",
        "secondary_metrics": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
        ],
        "test_used_for_selection": False,
    },
    "baseline_summary": aggregate_records,
    "quality_first_student": QUALITY_FIRST_STUDENT,
    "deployment_efficient_student": (
        DEPLOYMENT_EFFICIENT_STUDENT
    ),
    "teacher_families": teacher_families,
    "protocol_deviations": protocol_deviations,
    "teacher_checkpoint_policy": (
        "Exact model identifier, upstream revision, preprocessing, "
        "tokenizer and checkpoint SHA-256 must be locked in each "
        "teacher-specific notebook before full training."
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    TEACHER_ENTRY_CONTRACT_PATH,
    teacher_entry_contract,
)

cell_audit = {
    "schema_version": "phase3a_cell05_aggregate_entry_contract_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "student_count": len(aggregate_records),
    "seed_count_per_student": 3,
    "quality_first_student": QUALITY_FIRST_STUDENT,
    "deployment_efficient_student": (
        DEPLOYMENT_EFFICIENT_STUDENT
    ),
    "teacher_family_count": len(teacher_families),
    "entry_contract": str(TEACHER_ENTRY_CONTRACT_PATH),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
CELL05_AUDIT_PATH = AUDIT_DIR / (
    "phase3a_cell05_three_seed_aggregate_and_entry_contract_audit.json"
)
write_json(CELL05_AUDIT_PATH, cell_audit)

print("=" * 72)
print("THREE-SEED BASELINE SUMMARY")
print("=" * 72)
for row in ranked:
    print(
        f"#{row['rank_by_validation_cider_d']} "
        f"{row['student_id']} — {row['student_name']} | "
        f"CIDEr-D={row['cider_d_mean']:.8f} "
        f"+/- {row['cider_d_sample_sd']:.8f} | "
        f"BLEU-4={row['bleu_4_mean']:.8f} "
        f"+/- {row['bleu_4_sample_sd']:.8f}"
    )
print()
print("Quality-first candidate   :", QUALITY_FIRST_STUDENT)
print(
    "Deployment-efficient      :",
    DEPLOYMENT_EFFICIENT_STUDENT,
)
print("Teacher families          :", len(teacher_families))
for family_id, family in teacher_families.items():
    print(
        f"Teacher {family_id}                 : "
        f"{family['teacher']} -> {family['matched_student_id']}"
    )

print()
print("=" * 72)
print("PHASE 3A CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Three-seed statistics     : READY")
print("Teacher entry contract    : LOCKED")
print("Selection split           : Validation only")
print("Test Set status           : SEALED")
print("Audit                     :", CELL05_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_audit["elapsed_seconds"], 2),
    "giây",
)
print(
    "Next                      : package transition bundle and publish Phase 3 entry marker"
)


THREE-SEED BASELINE SUMMARY
#1 S3 — TinyViT-5M | CIDEr-D=1.07480359 +/- 0.00337928 | BLEU-4=0.33637524 +/- 0.00149546
#2 S2 — MobileViT-S | CIDEr-D=0.99322546 +/- 0.00947994 | BLEU-4=0.31510460 +/- 0.00437386
#3 S1 — MobileNetV3-Large | CIDEr-D=0.98583087 +/- 0.00678263 | BLEU-4=0.31102647 +/- 0.00247381

Quality-first candidate   : S3
Deployment-efficient      : S1
Teacher families          : 3
Teacher A                 : CLIP ViT-B/32 + shared caption decoder -> S1
Teacher B                 : BLIP caption teacher -> S2
Teacher C                 : SigLIP ViT-B/16 + shared caption decoder -> S3

PHASE 3A CELL 5 STATUS
Status                    : PASSED
Three-seed statistics     : READY
Teacher entry contract    : LOCKED
Selection split           : Validation only
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_pool_entry/audits/phase3a_cell05_three_seed_aggregate_and_entry_contract_audit.json
Elapsed                   : 0.01 giây
Next     

## Cell 6 — Đóng gói Phase 2 và phát hành Phase 3 entry marker

### Công việc

- kiểm kê artefact do notebook tạo;
- tạo manifest CSV, manifest JSON và SHA256SUMS;
- xác minh lại toàn bộ checksum;
- tạo `PHASE2_STUDENT_BASELINES_COMPLETE.json`;
- tạo `PHASE3_TEACHER_POOL_INPUT_READY.json`;
- khóa bước tiếp theo là Teacher A contract/preflight;
- không huấn luyện và không mở Test.

### Thời gian dự kiến

2–10 giây.

### Đầu ra dự kiến

- Phase 2 Student Baselines: COMPLETE;
- Phase 3 Teacher Pool Input: READY;
- manifest verification: ALL PASSED;
- Test Set: SEALED;
- có thể `Save Version` trên Kaggle.


In [6]:
CELL_STARTED_AT = time.perf_counter()

required_names = [
    "RUN_REGISTRY_PATH",
    "PROTOCOL_DEVIATION_PATH",
    "PER_RUN_CSV_PATH",
    "AGGREGATE_CSV_PATH",
    "TEACHER_ENTRY_CONTRACT_PATH",
    "run_records",
    "aggregate_records",
    "teacher_families",
]
assert not [
    name for name in required_names
    if name not in globals()
], "Hãy chạy Cell 1–5 trước."

PHASE2_COMPLETE_PATH = OUTPUT_ROOT / (
    "PHASE2_STUDENT_BASELINES_COMPLETE.json"
)
PHASE3_INPUT_READY_PATH = OUTPUT_ROOT / (
    "PHASE3_TEACHER_POOL_INPUT_READY.json"
)
MANIFEST_CSV_PATH = MANIFEST_DIR / (
    "phase3_teacher_pool_entry_bundle_manifest.csv"
)
MANIFEST_JSON_PATH = MANIFEST_DIR / (
    "phase3_teacher_pool_entry_bundle_manifest.json"
)
SHA256SUMS_PATH = MANIFEST_DIR / (
    "phase3_teacher_pool_entry_bundle_SHA256SUMS.txt"
)

excluded = {
    PHASE2_COMPLETE_PATH.resolve(),
    PHASE3_INPUT_READY_PATH.resolve(),
    MANIFEST_CSV_PATH.resolve(),
    MANIFEST_JSON_PATH.resolve(),
    SHA256SUMS_PATH.resolve(),
}

CELL06_AUDIT_PATH = AUDIT_DIR / (
    "phase3a_cell06_transition_packaging_audit.json"
)
pre_manifest_files = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
    and path.resolve() not in excluded
]
expected_manifest_artifacts = len(pre_manifest_files) + (
    0 if CELL06_AUDIT_PATH.exists() else 1
)
cell_audit = {
    "schema_version": "phase3a_cell06_packaging_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "expected_manifest_artifacts": (
        expected_manifest_artifacts
    ),
    "phase2_complete_marker_planned": str(
        PHASE2_COMPLETE_PATH
    ),
    "phase3_input_ready_marker_planned": str(
        PHASE3_INPUT_READY_PATH
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
    "teacher_training_started": False,
}
write_json(CELL06_AUDIT_PATH, cell_audit)

manifest_rows = []
for path in sorted(
    OUTPUT_ROOT.rglob("*"),
    key=lambda value: value.as_posix(),
):
    if not path.is_file():
        continue
    if path.resolve() in excluded:
        continue
    assert not path.name.endswith(".tmp")
    manifest_rows.append(
        {
            "relative_path": path.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
    )

assert manifest_rows
assert len(manifest_rows) == expected_manifest_artifacts
assert len({
    row["relative_path"]
    for row in manifest_rows
}) == len(manifest_rows)

with open(
    MANIFEST_CSV_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=[
            "relative_path",
            "size_bytes",
            "sha256",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

write_json(
    MANIFEST_JSON_PATH,
    {
        "schema_version": (
            "phase3_teacher_pool_entry_bundle_manifest_v1"
        ),
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "total_size_bytes": sum(
            row["size_bytes"]
            for row in manifest_rows
        ),
        "artifacts": manifest_rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
atomic_write_text(
    SHA256SUMS_PATH,
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
)

for row in manifest_rows:
    path = resolve_inside(
        OUTPUT_ROOT,
        row["relative_path"],
    )
    assert path.is_file()
    assert path.stat().st_size == row["size_bytes"]
    assert sha256_file(path) == row["sha256"]

manifest_files = {
    "csv": {
        "path": str(MANIFEST_CSV_PATH),
        "sha256": sha256_file(MANIFEST_CSV_PATH),
    },
    "json": {
        "path": str(MANIFEST_JSON_PATH),
        "sha256": sha256_file(MANIFEST_JSON_PATH),
    },
    "sha256sums": {
        "path": str(SHA256SUMS_PATH),
        "sha256": sha256_file(SHA256SUMS_PATH),
    },
}

phase2_complete = {
    "marker_version": (
        "phase2_student_baselines_complete_v1"
    ),
    "created_at_utc": now_utc(),
    "status": "complete_with_disclosed_deviation",
    "dataset": "MS COCO 2014 only",
    "split_protocol": "Karpathy",
    "student_ids": ["S1", "S2", "S3"],
    "seeds": [42, 123, 2026],
    "completed_run_count": len(run_records),
    "baseline_matrix_complete": True,
    "quality_first_student": QUALITY_FIRST_STUDENT,
    "deployment_efficient_student": (
        DEPLOYMENT_EFFICIENT_STUDENT
    ),
    "protocol_deviation_count": len(
        protocol_deviations
    ),
    "protocol_deviation_status": (
        "accepted_with_disclosure"
    ),
    "results": {
        "per_run_csv": str(PER_RUN_CSV_PATH),
        "aggregate_csv": str(AGGREGATE_CSV_PATH),
    },
    "manifest": manifest_files,
    "test_set_status": "sealed",
    "test_set_used": False,
    "ready_for_phase3": True,
}
write_json(PHASE2_COMPLETE_PATH, phase2_complete)

phase3_ready = {
    "marker_version": (
        "phase3_teacher_pool_input_ready_v1"
    ),
    "created_at_utc": now_utc(),
    "status": "ready",
    "phase2_complete_marker": {
        "path": str(PHASE2_COMPLETE_PATH),
        "sha256": sha256_file(PHASE2_COMPLETE_PATH),
    },
    "teacher_entry_contract": {
        "path": str(TEACHER_ENTRY_CONTRACT_PATH),
        "sha256": sha256_file(
            TEACHER_ENTRY_CONTRACT_PATH
        ),
    },
    "teacher_families": teacher_families,
    "authorized_next_action": (
        "Build Teacher A CLIP ViT-B/32 shared-decoder "
        "contract and run deterministic architecture/preprocessing "
        "preflight on Validation only."
    ),
    "teacher_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PHASE3_INPUT_READY_PATH, phase3_ready)

cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

print("=" * 72)
print("PHASE 2 -> PHASE 3 TRANSITION BUNDLE")
print("=" * 72)
print("Baseline runs             :", len(run_records), "/ 9")
print("Protocol deviations       :", len(protocol_deviations))
print("Manifest artifacts        :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Quality-first Student     :", QUALITY_FIRST_STUDENT)
print(
    "Deployment-efficient      :",
    DEPLOYMENT_EFFICIENT_STUDENT,
)

print()
print("=" * 72)
print("PHASE 3A CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 2 Baselines         : COMPLETE")
print("Phase 3 Teacher Input     : READY")
print("Teacher training started  : False")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Phase 2 marker            :", PHASE2_COMPLETE_PATH)
print(
    "Phase 2 marker SHA-256   :",
    sha256_file(PHASE2_COMPLETE_PATH),
)
print("Phase 3 marker            :", PHASE3_INPUT_READY_PATH)
print(
    "Phase 3 marker SHA-256   :",
    sha256_file(PHASE3_INPUT_READY_PATH),
)
print("Manifest CSV              :", MANIFEST_CSV_PATH)
print("Manifest JSON             :", MANIFEST_JSON_PATH)
print("SHA256SUMS                :", SHA256SUMS_PATH)
print("Audit                     :", CELL06_AUDIT_PATH)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : Save Version, then build Phase 3B Teacher A CLIP contract/preflight"
)


PHASE 2 -> PHASE 3 TRANSITION BUNDLE
Baseline runs             : 9 / 9
Protocol deviations       : 1
Manifest artifacts        : 11
Manifest verification     : ALL PASSED
Quality-first Student     : S3
Deployment-efficient      : S1

PHASE 3A CELL 6 STATUS
Status                    : PASSED
Phase 2 Baselines         : COMPLETE
Phase 3 Teacher Input     : READY
Teacher training started  : False
Test Parquet opened       : False
Test caption read         : False
Test images read          : False
Test metrics computed     : False
Test Set status           : SEALED
Test Set used             : False
Phase 2 marker            : /kaggle/working//phase3_teacher_pool_entry/PHASE2_STUDENT_BASELINES_COMPLETE.json
Phase 2 marker SHA-256   : 9c5532d63e87b7d5010318f8101ce0fba1341c27e44d3004fdaba707b2e87051
Phase 3 marker            : /kaggle/working//phase3_teacher_pool_entry/PHASE3_TEACHER_POOL_INPUT_READY.json
Phase 3 marker SHA-256   : 136f5d8eb0783fec74a37e21b87167226c5a0c4ed42a8eaf4f927fba26d6f